# 05. Torch Library Custom Ops and Fake Kernels

- **Category:** 12_compilation_and_performance
- **Lineage & Citations:** Cousot & Cousot (1977 *Abstract interpretation: a unified lattice model for static analysis of programs*); Ansel et al. (2024 *PyTorch 2: Faster Machine Learning Through Dynamic Python Bytecode Transformation and Graph Compilation*).
- **Purpose:** Formulate and verify custom operator registration via `torch.library`, abstract meta/fake shape kernels, and specialized forward-backward operators (L1 proximal soft-shrinkage, asymmetric affine quantization, and depthwise 1D convolution).
- **Topic Connections:** `🔗 Upstream Prerequisites:` [04. Torch Export AOT Autograd and IR Inspection](04_torch_export_aot_autograd_and_ir_inspection.ipynb) $\to$ `🔗 Downstream Syntheses:` [06. Memory Optimization Selective Checkpointing Offloading and Grad Hooks](06_memory_optimization_selective_checkpointing_offloading_and_grad_hooks.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not register custom ops for combinations of existing ATen primitives that Inductor can already fuse effectively; ensure every custom op has a registered fake meta-kernel to avoid compilation graph breaks.

## Mathematical Foundations & Operator Formulations

1. **L1 Proximal Soft-Shrinkage**:
   $$\operatorname{shrink}(x, \lambda) = \operatorname{sign}(x) \cdot \max(|x| - \lambda, 0)$$
2. **Asymmetric Affine Quantization**:
   $$s = \max\left(\frac{\max(x) - \min(x)}{255}, \epsilon\right), \quad z = \operatorname{clamp}\left(\operatorname{round}\left(-\frac{\min(x)}{s}\right), 0, 255\right)$$
   $$\hat{x} = \left( \operatorname{clamp}\left(\operatorname{round}\left(\frac{x}{s}\right) + z, 0, 255\right) - z \right) \cdot s$$
3. **Depthwise 1D Convolution**:
   $$y_{b, c, t} = \operatorname{bias}_c + \sum_{k=0}^{K-1} x_{b, c, t + k} \cdot w_{c, k}$$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/12_compilation_and_performance/05_torch_library_custom_ops_and_fake_kernels.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def custom_soft_shrink(x, lambd=0.5):
    return torch.sign(x) * torch.clamp(torch.abs(x) - float(lambd), min=0.0)

def custom_soft_shrink_meta(x_shape, dtype, device):
    return torch.empty(x_shape, dtype=dtype, device=device)

def custom_asymmetric_quantize(x, eps=1e-7):
    x_min = torch.amin(x, dim=-1, keepdim=True)
    x_max = torch.amax(x, dim=-1, keepdim=True)
    scale = torch.clamp((x_max - x_min) / 255.0, min=float(eps))
    zero_point = torch.clamp(torch.round(-x_min / scale), min=0.0, max=255.0)
    q = torch.clamp(torch.round(x / scale) + zero_point, min=0.0, max=255.0)
    return (q - zero_point) * scale, scale, zero_point

def custom_depthwise_conv1d(x, weight, bias):
    b, c, l = x.shape
    c_w, k = weight.shape
    out_len = l - k + 1
    y = torch.zeros(b, c, out_len, dtype=x.dtype, device=x.device)
    for t in range(out_len):
        y[:, :, t] = torch.sum(x[:, :, t : t + k] * weight.unsqueeze(0), dim=-1) + bias.unsqueeze(0)
    return y

In [ ]:
# Verify operators against golden reference tensors
shrink = custom_soft_shrink(weights["shrink_x"], lambd=0.5)
assert torch.max(torch.abs(shrink - weights["shrink_ref"])) < 1e-5

x_hat, scale, zero_pt = custom_asymmetric_quantize(weights["quant_x"], eps=1e-7)
assert torch.max(torch.abs(x_hat - weights["quant_x_hat"])) < 1e-5
assert torch.max(torch.abs(scale - weights["quant_scale"])) < 1e-5
assert torch.max(torch.abs(zero_pt - weights["quant_zero"])) < 1e-5

conv = custom_depthwise_conv1d(weights["conv_x"], weights["conv_w"], weights["conv_b"])
assert torch.max(torch.abs(conv - weights["conv_ref"])) < 1e-5
print("Custom ops parity verification passed!")

In [ ]:
# Microbenchmark: Custom operator evaluation latency
t0 = time.perf_counter()
for _ in range(500):
    _ = custom_soft_shrink(weights["shrink_x"], 0.5)
    _ = custom_asymmetric_quantize(weights["quant_x"], 1e-7)
    _ = custom_depthwise_conv1d(weights["conv_x"], weights["conv_w"], weights["conv_b"])
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Combined custom ops latency: {lat_us:.2f} us")

In [ ]:
# Meta/Fake Kernel Verification: Abstract shape inference without storage
meta_out = custom_soft_shrink_meta(weights["shrink_x"].shape, weights["shrink_x"].dtype, "meta")
print(f"Meta tensor device: {meta_out.device}, shape: {meta_out.shape}")
assert meta_out.is_meta, "Meta output must be a virtual meta tensor!"
assert meta_out.shape == weights["shrink_x"].shape, "Shape must match input!"

# Sparsity check: Verify soft shrinkage zeroes out sub-threshold values
sub_threshold = torch.tensor([-0.4, 0.2, -0.1, 0.49])
shrunk = custom_soft_shrink(sub_threshold, lambd=0.5)
assert torch.all(shrunk == 0.0), "All sub-threshold values must shrink to zero!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **`torch.library`** provides a first-class extension mechanism that registers custom kernels with explicit type schemas across all compilation phases.
2. **Meta / Fake Kernels** infer output dimensions and strides symbolically, allowing ahead-of-time tracing without GPU device allocations.
3. **Autograd Registration** pairs custom forward kernels with analytical backward adjoints for seamless automatic differentiation.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [05_torch_library_custom_ops_and_fake_kernels.md](../../markdown/12_compilation_and_performance/05_torch_library_custom_ops_and_fake_kernels.md)
- **PyTorch 2.x Module:** [05_torch_library_custom_ops_and_fake_kernels.py](../../pytorch2/12_compilation_and_performance/05_torch_library_custom_ops_and_fake_kernels.py)
- **Functional JAX Module:** [05_torch_library_custom_ops_and_fake_kernels.py](../../jax/12_compilation_and_performance/05_torch_library_custom_ops_and_fake_kernels.py)
- **Native Rust Candle Track:** [05_torch_library_custom_ops_and_fake_kernels.rs](../../candle/12_compilation_and_performance/05_torch_library_custom_ops_and_fake_kernels.rs)
- **Hardware Kernel (CPU / WGSL):** [05_torch_library_custom_ops_and_fake_kernels.rs](../../kernels/12_compilation_and_performance/05_torch_library_custom_ops_and_fake_kernels.rs)
- **Golden SafeTensors:** [05_torch_library_custom_ops_and_fake_kernels.safetensors](../../weights/12_compilation_and_performance/05_torch_library_custom_ops_and_fake_kernels.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 04. Torch Export AOT Autograd and IR Inspection](04_torch_export_aot_autograd_and_ir_inspection.ipynb) | [Master Index](../../README.md#12_compilation_and_performance) | [Next: 06. Memory Optimization Selective Checkpointing Offloading and Grad Hooks](06_memory_optimization_selective_checkpointing_offloading_and_grad_hooks.ipynb) &rarr;